# utils_test_gaussian_policy

Gaussian policy test (mu, log_sigma) with LTspice parity check.

This notebook (utils_test_gaussian_policy) tests a Gaussian policy (mu, log_sigma) using pytorch2ltspice utilities.
It builds a small PyTorch model, exports an LTspice subcircuit, and validates
parity between PyTorch and LTspice outputs across sampled inputs.
This notebook serves as a focused regression test and a usage example for Gaussian policies.

---

## Quick Start

1. Set `ENV_NAME` and `MODEL_NAME` in the Configuration section.
2. Run `step1()` to generate the model class and subcircuit.
3. Run `step2()` and `step3()` to simulate and compare outputs.

---



## Change Log:

2026-01-04,
- Added notebook description and change log for gaussian policy test.



In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from torch import nn

from PyLTSpice import LTspice, RawRead, SimRunner

from pytorch2ltspice import export_model_to_ltspice
from pytorch2ltspice.utils.modelgen import build_model_from_sequential
from pytorch2ltspice.utils.sampling import sample_on_clock
from pytorch2ltspice.utils.siggen import generate_siggen_asc_asy

---
## Configuration

In [ ]:
# Environment configuration
# Add entries to ENV_CONFIG with required keys: "in" and "out"
ENV_NAME = "env_buck_9x2_ppo"              # Select environment file here
ENV_CONFIG = {
    "env_buck_9x2_ppo":  {
        "in": 9,
        "out": 2,
        "output_activation": ["tanh", "identity"],
        "output_mask": [True, False],
    },
    # Add more environments as needed
}

# Model configuration
# Add entries to MODEL_CONFIG with required key: "clk_needed"
MODEL_NAME = "mlp"                     # "mlp"/"rnn_linear"/"gru_linear"/"linear_lstm_linear"
MODEL_CONFIG = {
    "mlp":  {"clk_needed": False},
    "rnn_linear":  {"clk_needed": True},
    "gru_linear":  {"clk_needed": True},
    "linear_lstm_linear":  {"clk_needed": True},
    # Add more models as needed
}

# Output activation for Gaussian policy (mu, log_sigma)
OUTPUT_ACTIVATION = ENV_CONFIG[ENV_NAME]["output_activation"]
OUTPUT_MASK = ENV_CONFIG[ENV_NAME]["output_mask"]

# LTspice simulation configuration
SIM_STEP = 200                         # Number of time steps to run in LTSpice
SIM_TIMEOUT = 300                      # Timeout therhold in seconds

# LTspice schematic uses NNIN1..NNIN9 (NNIN9 is unused by the model).
LTSPICE_INPUT_DIM = 9

# Working directory for pyLTspice
NOTEBOOK_DIR = Path.cwd()
ENVDIR  = NOTEBOOK_DIR / "gym"
OUTDIR  = NOTEBOOK_DIR / "gym"         # save at same directory as .asc file
WORKDIR = NOTEBOOK_DIR / "tmp"
WORKDIR.mkdir(exist_ok=True)

DEVICE = torch.device('cpu')

---
## Model Definition

In [ ]:
def make_model(model_name: str) -> nn.Sequential:
    if model_name == "mlp":
        return nn.Sequential(
            nn.Linear(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
        )
    elif model_name == "rnn_linear":
        return nn.Sequential(
            nn.RNNCell(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
        )
    elif model_name == "gru_linear":
        return nn.Sequential(
            nn.GRUCell(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
        )
    elif model_name == "linear_lstm_linear":
        return nn.Sequential(
            nn.Linear(ENV_CONFIG[ENV_NAME]["in"], 32),
            nn.Tanh(),
            nn.LSTMCell(32, 32),
            nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
        )
    # Example extension for stacked LSTM cells:
    # elif model_name == "stacked_lstm":
    #     return nn.Sequential(
    #         nn.LSTMCell(ENV_CONFIG[ENV_NAME]["in"], 32),
    #         nn.LSTMCell(32, 32),
    #         nn.Linear(32, ENV_CONFIG[ENV_NAME]["out"]),
    #     )
    else:
        raise ValueError(f"Unknown model preset: {model_name}")

---
## Step1) Create Python code and LTspice sub-circuit

In [ ]:
def step1() -> nn.modules:
    # Select input sequential model
    seq = make_model(MODEL_NAME)

    # Generate & save python code via utils
    GenClass = build_model_from_sequential(
        MODEL_NAME,
        seq,
        out_dir=ENVDIR,
        out_py_name=PY_FILENAME,
        output_activation=OUTPUT_ACTIVATION,
    )

    # Instantiate the generated class
    actor = GenClass().to(DEVICE)

    # Export LTSpice sub-circuit (full + mask)
    lt_inputs = [f"NNIN{i+1}" for i in range(LTSPICE_INPUT_DIM)]
    export_model_to_ltspice(
        actor.model,
        filename=f"{OUTDIR}/{SP_FILENAME}.sp",
        subckt_name=MODEL_NAME,
        input_ports=lt_inputs,
        output_activation=OUTPUT_ACTIVATION,
        verbose=False,
    )
    export_model_to_ltspice(
        actor.model,
        filename=f"{OUTDIR}/{SP_FILENAME_LITE}.sp",
        subckt_name=MODEL_NAME,
        input_ports=lt_inputs,
        output_activation=OUTPUT_ACTIVATION,
        output_mask=OUTPUT_MASK,
        verbose=False,
    )

    return actor

## Step2) Run Simulation on LTspice and Python

In [ ]:
def step2(module, use_mask: bool):
    out_count = ENV_CONFIG[ENV_NAME]["out"]
    mask_indices = [i for i, m in enumerate(OUTPUT_MASK) if m]
    if use_mask:
        selected_indices = mask_indices
    else:
        selected_indices = list(range(out_count))

    # 0) Create random noise generator asc/asy (noise for log_sigma)
    NOISE_SIGMA = 0.0 if use_mask else 1.0
    noise_indices = [out_count - 1] if out_count > 1 else [0]
    noises = np.zeros((SIM_STEP, len(noise_indices)), dtype=float)
    for out_idx, i in enumerate(noise_indices):
        noises[:, out_idx] = np.random.normal(loc=0.0, scale=NOISE_SIGMA, size=SIM_STEP)
        generate_siggen_asc_asy(
            signals=noises[:, out_idx],
            asc_path=f"{ENVDIR}/sig_gen{i+1}.asc",
            gen_symbol=True,
            subckt_name=f"sig_gen{i+1}",
        )

    # 1) Create parameter file
    with open(f"{ENVDIR}/{ENV_NAME}_param.txt", "w", encoding="utf-8") as f:
        f.write(f".param STEPS={SIM_STEP}\n")
        nn_inputs = " ".join(f"NNin{i+1}" for i in range(LTSPICE_INPUT_DIM))
        nn_outputs = " ".join(f"NNout{i+1}" for i in selected_indices)
        ports = " ".join(p for p in [nn_inputs, ("ctrlclk" if MODEL_CONFIG[MODEL_NAME]["clk_needed"] else ""), nn_outputs] if p)
        f.write(f"X99 {ports} {MODEL_NAME}\n")
        sp_filename = SP_FILENAME_LITE if use_mask else SP_FILENAME
        f.write(f".include {sp_filename}.sp\n")

    # 2) Create PyLTspice SimRunner instance at WORKDIR
    shutil.copy2(f"{OUTDIR}/{sp_filename}.sp", f"{WORKDIR}/")
    shutil.copy2(f"{ENVDIR}/{ENV_NAME}_param.txt", f"{WORKDIR}/")
    runner = SimRunner(output_folder=WORKDIR, simulator=LTspice)
    netlist = runner.create_netlist(f"{ENVDIR}/{ENV_NAME}.asc")

    # 3) Run LTSpice simulation
    raw, log = runner.run_now(netlist, timeout=SIM_TIMEOUT)
    raw_data = RawRead(raw)
    df = raw_data.to_dataframe()
    df = sample_on_clock(df, clk="V(ctrlclk)")

    # 4) Extract states, actions
    states = df[[f"V(nnin{i+1})" for i in range(LTSPICE_INPUT_DIM)]].values[:-1]
    nnouts = df[[f"V(nnout{i+1})" for i in selected_indices]].values[:-1]
    actions = df[[f"V(nnout{i+1}n)" for i in mask_indices]].values[:-1]

    # 5) Clean PyLTspice files
    runner.cleanup_files()
    os.remove(f"{ENVDIR}/{ENV_NAME}.net")
    os.remove(f"{WORKDIR}/{sp_filename}.sp")
    os.remove(f"{WORKDIR}/{ENV_NAME}_param.txt")

    # 6) Calculate PyTorch output using observation from LTspice
    states_py = states[:, :ENV_CONFIG[ENV_NAME]["in"]]
    states_t = torch.tensor(states_py, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        nnouts_py_full = module(states_t).cpu().numpy()
        if nnouts_py_full.ndim == 1:
            nnouts_py_full = nnouts_py_full.reshape(-1, out_count)
        nnouts_py = nnouts_py_full[:, selected_indices]

    noises = noises[:nnouts_py_full.shape[0]]
    if out_count > 1:
        noise = noises[:, :1]
        actions_py = nnouts_py_full[:, [0]] + np.exp(nnouts_py_full[:, [1]]) * noise
    else:
        actions_py = nnouts_py_full[:, [0]] + noises

    return nnouts, nnouts_py, actions, actions_py, selected_indices


## Step3) Compare outputs from PyTorch and LTspice

In [ ]:
def step3(nnouts, nnouts_py, actions, actions_py, selected_indices, use_mask: bool):
    #Plot Scatter graph
    fig = go.Figure()
    ltspice_o = np.asarray(nnouts)
    pytorch_o = np.asarray(nnouts_py)
    ltspice_on = np.asarray(actions)
    pytorch_on = np.asarray(actions_py)
    samples = ltspice_on.shape[0]
    x_axis = np.arange(samples)
    for local_idx, orig_idx in enumerate(selected_indices):
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=ltspice_o[:, local_idx],
            mode="markers",
            name=f"NNOUT{orig_idx + 1}(LTspice)"
        ))
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=pytorch_o[:, local_idx],
            mode="markers",
            name=f"NNOUT{orig_idx + 1}(PyTorch)"
        ))
    for local_idx, orig_idx in enumerate([i for i, m in enumerate(OUTPUT_MASK) if m]):
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=ltspice_on[:, local_idx],
            mode="markers",
            name=f"NNOUT{orig_idx + 1}n(LTspice)"
        ))
        fig.add_trace(go.Scatter(
            x=x_axis,
            y=pytorch_on[:, local_idx],
            mode="markers",
            name=f"NNOUT{orig_idx + 1}n(PyTorch)"
        ))
    mask_label = "mask" if use_mask else "full"
    fig.update_layout(
        title=f"ENV={ENV_NAME}<br>MODEL={MODEL_NAME} ({mask_label})",
        xaxis_title="Sample index",
        yaxis_title="Output"
    )
    fig.show()

    #Print MAE/MSE
    diff = nnouts - nnouts_py
    diffn = actions - actions_py
    mae_per_output = np.mean(np.abs(diff), axis=0)
    mse_per_output = np.mean(diff ** 2, axis=0)
    mae_per_outputn = np.mean(np.abs(diffn), axis=0)
    mse_per_outputn = np.mean(diffn ** 2, axis=0)
    output_indices = selected_indices
    noise_indices = [i for i, m in enumerate(OUTPUT_MASK) if m]
    for local_idx, (mae_val, mse_val) in enumerate(zip(mae_per_output, mse_per_output)):
        orig_idx = output_indices[local_idx]
        print(f"  NNOUT{orig_idx + 1}: MAE={mae_val:.6f}, MSE={mse_val:.6f}")
    for local_idx, (mae_valn, mse_valn) in enumerate(zip(mae_per_outputn, mse_per_outputn)):
        orig_idx = noise_indices[local_idx]
        print(f"  NNOUT{orig_idx + 1}n: MAE={mae_valn:.6f}, MSE={mse_valn:.6f}")


---
## Execution

In [ ]:
PY_FILENAME = ENV_NAME + "_" + MODEL_NAME + "_gaussian"     # output python file name
SP_FILENAME = ENV_NAME + "_" + MODEL_NAME + "_gaussian"     # output ltspice subcircuit name
SP_FILENAME_LITE = SP_FILENAME + "_lite"
actor = step1()
nnouts, nnouts_py, actions, actions_py, selected_indices = step2(actor, use_mask=False)
step3(nnouts, nnouts_py, actions, actions_py, selected_indices, use_mask=False)
nnouts, nnouts_py, actions, actions_py, selected_indices = step2(actor, use_mask=True)
step3(nnouts, nnouts_py, actions, actions_py, selected_indices, use_mask=True)